# Notebook 01 — QC, Filtrado y Normalización

**Entrada:** GIN71_filtered_feature_bc_matrix.h5 (Space Ranger)  
**Salida:** GIN71_filtered_normalized.h5ad 


## 1. Librerías y configuración


In [ ]:
import scanpy as sc
import squidpy as sq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import os

# Parámetro principal: cada sample_id corresponde a un paciente. 
SAMPLE_ID = 'GIN71'

# Rutas
DATA_PATH       = f'/mnt/16TB/MAC/Documents/Projects/TFM_Irati/OUTPUTS/{SAMPLE_ID}'
PATOLOGOS_PATH  = f'/home/imartinezle/Spatial_transcriptomics/Anotaciones_Patologos/{SAMPLE_ID}_layer_normalizado.csv'
OUTPUT_FILE     = f'/home/imartinezle/h5ad_outputs/{SAMPLE_ID}_filtered_normalized.h5ad'
FIGDIR          = f'/home/imartinezle/Figuras/figuras_{SAMPLE_ID}/Filtrado_normalizado'

os.makedirs(FIGDIR, exist_ok=True)
sc.settings.figdir = FIGDIR
sc.settings.set_figure_params(dpi=150, facecolor='white')

print(f'Procesando paciente: {SAMPLE_ID}')
print(f'Scanpy:  {sc.__version__}')
print(f'Squidpy: {sq.__version__}')


## 2. Cargar datos de Space Ranger

`sc.read_visium()` carga automáticamente la matriz de conteos,
las coordenadas espaciales de los spots y la imagen histológica.


In [ ]:
adata = sc.read_visium(
    path=DATA_PATH,
    count_file='filtered_feature_bc_matrix.h5',
    load_images=True
)
adata.var_names_make_unique()

print(f'Spots: {adata.n_obs}')
print(f'Genes: {adata.n_vars}')
print(adata.obs.head())


## 3. Visualización inicial sobre el tejido


In [ ]:
sq.pl.spatial_scatter(
    adata,
    library_id=SAMPLE_ID,
    img=True,
    img_res_key='hires',
    color='in_tissue',
    size=1,
    alpha=0.8,
    legend_loc=None,
    title=f'{SAMPLE_ID} — Spots on tissue'
)
plt.savefig(f'{FIGDIR}01_spots_tissue.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Métricas de calidad (QC)

Calculamos por spot:
- **n_genes_by_counts**: número de genes detectados
- **total_counts**: total de UMIs
- **pct_counts_mt**: % de reads mitocondriales (valores altos indican
  células dañadas o muertas)


In [ ]:
# Identificar genes mitocondriales (empiezan por MT-)
adata.var['mt'] = adata.var_names.str.startswith('MT-')

sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)

print(adata.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].describe())


### 4.1 Violines de QC


In [ ]:
# Violines con matplotlib
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f'QC metrics — {SAMPLE_ID}', fontsize=14, fontweight='bold')

axes[0].violinplot(adata.obs['n_genes_by_counts'], showmedians=True)
axes[0].set_title('Genes per spot')
axes[0].set_ylabel('N genes')
axes[0].set_xticks([])

axes[1].violinplot(adata.obs['total_counts'], showmedians=True)
axes[1].set_title('Total UMIs per spot')
axes[1].set_ylabel('Total counts')
axes[1].set_xticks([])

axes[2].violinplot(adata.obs['pct_counts_mt'], showmedians=True)
axes[2].set_title('% Mitochondrial genes')
axes[2].set_ylabel('% MT')
axes[2].set_xticks([])

plt.tight_layout()
plt.savefig(f'{FIGDIR}02_QC_violins.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Violines con scanpy (incluye distribución de puntos)
sc.pl.violin(
    adata,
    keys=['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    multi_panel=True,
    stripplot=True,
    jitter=0.2,
    size=2.5,
    density_norm='width',
    save=f'_QC_{SAMPLE_ID}.png'
)


### 4.2 QC espacial

Visualizar las métricas sobre el tejido para detectar si los spots
de baja calidad se concentran en alguna región concreta.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(f'Spatial QC — {SAMPLE_ID}', fontsize=14, fontweight='bold')

sc.pl.spatial(adata, color='n_genes_by_counts', ax=axes[0], show=False, title='Genes per spot')
sc.pl.spatial(adata, color='total_counts',      ax=axes[1], show=False, title='Total UMIs')
sc.pl.spatial(adata, color='pct_counts_mt',     ax=axes[2], show=False, title='% Mitochondrial')

plt.tight_layout()
plt.savefig(f'{FIGDIR}03_QC_spatial.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Filtrado

Umbrales aplicados (ajustar según los violines de arriba):
- Spots con < 200 genes detectados → eliminados (poco informativos)
- Spots con > 10% de reads mitocondriales → eliminados (células dañadas)
- Genes expresados en < 3 spots → eliminados (ruido)


In [ ]:
print(f'Spots antes del filtrado: {adata.n_obs}')
print(f'Genes antes del filtrado: {adata.n_vars}')

# --- Umbrales (ajustar según violines) ---
MIN_GENES  = 200   # mínimo de genes por spot
MAX_PCT_MT = 10    # máximo % mitocondrial
MIN_SPOTS  = 3     # mínimo de spots en que debe expresarse un gen
# -----------------------------------------

sc.pp.filter_cells(adata, min_genes=MIN_GENES)
adata = adata[adata.obs['pct_counts_mt'] < MAX_PCT_MT].copy()
sc.pp.filter_genes(adata, min_cells=MIN_SPOTS)

print(f'Spots después del filtrado: {adata.n_obs}')
print(f'Genes después del filtrado: {adata.n_vars}')

# Violines post-filtrado para comparar con los anteriores
sc.pl.violin(
    adata,
    keys=['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    multi_panel=True,
    stripplot=True,
    jitter=0.2,
    size=2.5,
    density_norm='width',
    save=f'_QC_postfilt_{SAMPLE_ID}.png'
)


## 6. Normalización

- **normalize_total**: normaliza cada spot a 10.000 counts (equivalente a CPM)
- **log1p**: aplica log(x+1) para estabilizar la varianza
- Los counts crudos se guardan en `adata.layers['counts']` para uso posterior
  (por ejemplo, en InferCNV o RCTD, que requieren datos sin normalizar)


In [ ]:
# Guardar counts crudos antes de normalizar
adata.layers['counts'] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Identificar genes altamente variables (HVGs) — usados en PCA y clustering
sc.pp.highly_variable_genes(adata, flavor='seurat', n_top_genes=3000)

print(f'Genes altamente variables (HVGs): {adata.var.highly_variable.sum()}')

sc.pl.highly_variable_genes(adata, save=f'_HVG_{SAMPLE_ID}.png')


## 7. Reducción de dimensionalidad y UMAP

PCA + grafo de vecinos + UMAP. El clustering (Leiden, Louvain, GraphST,
BayesSpace) se realizará en el Notebook 02.


In [ ]:
sc.pp.scale(adata, max_value=10)

# PCA
sc.tl.pca(adata, svd_solver='arpack', use_highly_variable=True)
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True,
                         save=f'_PCA_variance_{SAMPLE_ID}.png')

# Grafo de vecinos
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)

# UMAP
sc.tl.umap(adata)

sc.pl.umap(
    adata,
    color=['total_counts', 'n_genes_by_counts', 'pct_counts_mt'],
    ncols=3,
    save=f'_UMAP_QC_{SAMPLE_ID}.png'
)

print('PCA y UMAP completados')


## 8. Anotaciones del patólogo

Se carga el CSV normalizado generado en el Notebook 00 y se añade la
columna `Layer_patologo` al objeto. Los spots unannotated se pueden eliminar.


In [ ]:
# Cargar anotaciones normalizadas (generadas en el Notebook 00)
informacion_patologos = pd.read_csv(PATOLOGOS_PATH, sep='\t')
informacion_patologos = informacion_patologos.set_index('Cell')

# Añadir al objeto: reindexar por barcode para asegurar alineación correcta
adata.obs['Layer_patologo'] = informacion_patologos.reindex(adata.obs_names)['Layer']
adata.obs['Layer_patologo'] = adata.obs['Layer_patologo'].astype('category')

print('Layer_patologo añadido:', 'Layer_patologo' in adata.obs.columns)
print(adata.obs['Layer_patologo'].value_counts())


In [ ]:
# ELIMINAR LOS SPOTS UNANNOTATED PARA QUE NO AFECTEN A LOS RESULTADOS DE LOS ANÁLISIS
adata = adata[adata.obs['Layer_patologo'] != 'unannotated']

### 8.1 Visualización de anotaciones del patólogo


In [ ]:
# Paleta de colores unificada para todo el pipeline
PALETTE_PATOLOGO = {
    'tumor':         '#D85A30',
    'stroma':        '#1D9E75',
    'stroma_linfos': '#534AB7',
    'unannotated':   '#B4B2A9',
}

# Construir ListedColormap en el mismo orden que las categorías del objeto
categorias    = adata.obs['Layer_patologo'].cat.categories.tolist()
lista_colores = [PALETTE_PATOLOGO[cat] for cat in categorias]
cmap_patologo = ListedColormap(lista_colores)

print('Categorías:', categorias)

# Sin imagen de fondo
sq.pl.spatial_scatter(
    adata,
    color='Layer_patologo',
    palette=cmap_patologo,
    img=False,
    shape='hex',
    size=1.5,
    frameon=False,
    legend_loc='right margin',
    title=f'Pathologists annotation — {SAMPLE_ID}'
)
plt.savefig(f'{FIGDIR}04_pathologist_annotation.png', dpi=150, bbox_inches='tight')
plt.show()

# Con imagen de fondo
sq.pl.spatial_scatter(
    adata,
    color='Layer_patologo',
    palette=cmap_patologo,
    library_id=SAMPLE_ID,
    img=True,
    shape='hex',
    size=1.5,
    frameon=False,
    legend_loc='right margin',
    title=f'Pathologists annotation — {SAMPLE_ID}'
)
plt.savefig(f'{FIGDIR}05_pathologist_annotation_img.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Guardar objeto


In [ ]:
adata.write_h5ad(OUTPUT_FILE)

print(f'Objeto guardado en: {OUTPUT_FILE}')
print(f'Spots: {adata.n_obs}')
print(f'Genes: {adata.n_vars}')
print(adata)
